In [1]:
import os
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "2"
import numpy as np
import pandas as pd
import tensorflow as tf
import keras
from dataclasses import dataclass
from pathlib import Path
from cnnClassifier import logger
from cnnClassifier.constants import *
from cnnClassifier.utils.common import read_yaml, load_json, load_image

In [2]:
%pwd

'/home/nima/Projects/ML-Project-Casting_Classification/Project/research'

In [3]:
os.chdir("../")

In [4]:
%pwd

'/home/nima/Projects/ML-Project-Casting_Classification/Project'

In [ ]:
@dataclass(frozen=True)
class PredictionConfig:
    trained_model_path: Path
    metadata_path: Path


class ConfigurationManager:
    def __init__(self, config_filepath=CONFIG_FILE_PATH):
        self.config = read_yaml(config_filepath)

    def get_prediction_config(self) -> PredictionConfig:
        c = self.config.evaluation      # same artifacts the evaluation stage produced
        return PredictionConfig(trained_model_path=Path(c.trained_model_path),
                                metadata_path=Path(c.metadata_path))

In [6]:
class Predictor:
    def __init__(self, config: PredictionConfig):
        meta = load_json(config.metadata_path)
        self.threshold = float(meta["threshold"])
        self.img_size = tuple(meta["input"]["size"])
        self.idx_to_class = {int(v): k for k, v in meta["class_to_idx"].items()}
        self.model = keras.models.load_model(config.trained_model_path, compile=False)
        logger.info(f"predictor ready | threshold={self.threshold:.3f} | input={self.img_size}")

    def predict(self, paths, batch_size: int = 32) -> pd.DataFrame:
        """Classify one or many image files; returns P(defect), the label and the class name."""
        paths = [str(p) for p in paths]
        if not paths:
            raise ValueError("no image paths given")
        ds = (tf.data.Dataset.from_tensor_slices(paths)
              .map(lambda p: load_image(p, self.img_size), num_parallel_calls=tf.data.AUTOTUNE)
              .batch(batch_size).prefetch(tf.data.AUTOTUNE))
        probs = self.model.predict(ds, verbose=0).ravel()
        labels = (probs >= self.threshold).astype(int)
        return pd.DataFrame({
            "filepath": paths,
            "p_defect": probs.round(4),
            "label": labels,
            "prediction": [self.idx_to_class[int(l)] for l in labels],
        })

    def predict_folder(self, folder, extns=(".jpg", ".jpeg", ".png", ".bmp")) -> pd.DataFrame:
        files = sorted(p for p in Path(folder).iterdir() if p.suffix.lower() in extns)
        return self.predict(files)

In [7]:
config = ConfigurationManager()
predictor = Predictor(config.get_prediction_config())

test_df = pd.read_csv("artifacts/splits/test.csv")
result = predictor.predict(test_df["filepath"].head(5))
result

[2026-11-04 12:37:00,194: INFO: common: yaml file: config/config.yaml loaded successfully]
[2026-11-04 12:37:00,196: INFO: common: json file loaded succesfully from: artifacts/evaluation/model_metadata.json]
[2026-11-04 12:37:02,543: INFO: 3152301022: predictor ready | threshold=0.485 | input=(224, 224)]


,filepath,p_defect,label,prediction
0,artifacts/data_ingestion/casting_512x512/casti...,1.0000,1,def_front
1,artifacts/data_ingestion/casting_512x512/casti...,1.0000,1,def_front
2,artifacts/data_ingestion/casting_512x512/casti...,0.0003,0,ok_front
3,artifacts/data_ingestion/casting_512x512/casti...,1.0000,1,def_front
4,artifacts/data_ingestion/casting_512x512/casti...,1.0000,1,def_front


In [8]:
full = predictor.predict(test_df["filepath"])
print("accuracy via predictor:", (full["label"].to_numpy() == test_df["label"].to_numpy()).mean())

accuracy via predictor: 1.0


In [9]:
ref = np.load("artifacts/evaluation/test_probs.npy")
print("max |diff|:", np.abs(ref - full["p_defect"].to_numpy()).max())   # should be below ~1e-4 (p_defect is rounded to 4 decimals)

max |diff|: 4.9710274e-05
